In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# ==============================
# 1. إنشاء Spark Session
# ==============================
spark = SparkSession.builder \
    .appName("Flight Data Warehouse") \
    .getOrCreate()

# ==============================
# 2. تحميل البيانات الخام (Bronze)
# ==============================
flights_df = spark.read.csv(
    "/home/ahmed-refat/Desktop/NYC_delays.csv",
    header=True,
    inferSchema=True
)

# ==============================
# 3. تنظيف البيانات (Silver)
# ==============================

# تحويل تاريخ الرحلة من string لـ date + حذف الأعمدة الغير مفيدة
flights_clean = flights_df \
    .withColumn("FL_DATE", F.to_date(F.col("FL_DATE"), "M/d/yyyy hh:mm:ss a")) \
    .drop(
        "ORIGIN_AIRPORT_SEQ_ID", "ORIGIN_CITY_MARKET_ID",
        "DEST_AIRPORT_SEQ_ID", "DEST_CITY_MARKET_ID",
        "ARR_DELAY_GROUP", "TOTAL_ADD_GTIME", "LONGEST_ADD_GTIME",
        "DISTANCE_GROUP", "ARR_TIME_BLK", "DEP_DELAY_NEW", 
        "ARR_DELAY_NEW", "FLIGHTS"
    )

# تحويل CANCELLED و DIVERTED من double لـ integer
flights_clean = flights_clean \
    .withColumn("CANCELLED", F.col("CANCELLED").cast("integer")) \
    .withColumn("DIVERTED", F.col("DIVERTED").cast("integer"))

# ملء قيم الـ NULL في أعمدة التأخير بـ 0
# لأن الـ NULL بتعني إن مفيش تأخير من هذا النوع
flights_clean = flights_clean \
    .fillna(0, subset=[
        "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY",
        "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY",
        "DEP_DELAY", "ARR_DELAY"
    ])

# إضافة أعمدة تشير إذا كانت الرحلة متأخرة في الإقلاع أو الوصول
flights_clean = flights_clean \
    .withColumn("IS_DEP_DELAYED", F.when(F.col("DEP_DELAY") > 0, 1).otherwise(0)) \
    .withColumn("IS_ARR_DELAYED", F.when(F.col("ARR_DELAY") > 0, 1).otherwise(0))

# حذف الصفوف المكررة
flights_clean = flights_clean.dropDuplicates()

# تحويل أوقات الإقلاع والوصول من integer (مثل 659) لـ string بصيغة HH:mm
flights_clean = flights_clean \
    .withColumn("CRS_DEP_TIME", F.lpad(F.col("CRS_DEP_TIME").cast("string"), 4, "0")) \
    .withColumn("CRS_DEP_TIME", F.to_timestamp(F.col("CRS_DEP_TIME"), "HHmm")) \
    .withColumn("CRS_DEP_TIME", F.col("CRS_DEP_TIME").cast("string").substr(12, 5)) \
    .withColumn("CRS_ARR_TIME", F.lpad(F.col("CRS_ARR_TIME").cast("string"), 4, "0")) \
    .withColumn("CRS_ARR_TIME", F.to_timestamp(F.col("CRS_ARR_TIME"), "HHmm")) \
    .withColumn("CRS_ARR_TIME", F.col("CRS_ARR_TIME").cast("string").substr(12, 5))

# ==============================
# 4. عرض النتيجة للتحقق
# ==============================
flights_clean.printSchema()
flights_clean.show(3)

+----------+-----------------+-----------------+-----------------+------+----------------+---------------+---------------+----+-------------------+-------------+------------+--------+---------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+---------+--------------+-------------------+--------------+--------------+
|   FL_DATE|OP_UNIQUE_CARRIER|OP_CARRIER_FL_NUM|ORIGIN_AIRPORT_ID|ORIGIN|ORIGIN_CITY_NAME|ORIGIN_STATE_NM|DEST_AIRPORT_ID|DEST|     DEST_CITY_NAME|DEST_STATE_NM|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|CANCELLED|CANCELLATION_CODE|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|IS_DEP_DELAYED|IS_ARR_DELAYED|
+----------+-----------------+-----------------+-----------------+------+----------------+---------------+---------------+----+-------------------+-----------

In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("processing_2_weather_data") \
    .getOrCreate()

# لود أول داتا ست
weather_df = spark.read.csv("/home/ahmed-refat/Desktop/flights & airports/Raw Data(batch)/weather.csv", header=True, inferSchema=True)

weather_clean = weather_df \
    .drop("feelslike", "dew", "solarradiation", "solarenergy", 
          "uvindex", "icon", "stations", "name") \
    .fillna(0.0, subset=["precip", "snow", "snowdepth", "windgust", "precipprob"]) \
    .fillna("None", subset=["preciptype", "severerisk"]) \
    .withColumn("datetime", F.col("datetime").cast("timestamp")) \
    .withColumn("date", F.to_date(F.col("datetime"))) \
    .withColumn("hour", F.hour(F.col("datetime"))) \
    .dropDuplicates()

weather_clean.printSchema()
weather_clean.show(3)

root
 |-- datetime: timestamp (nullable = true)
 |-- temp: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- precip: double (nullable = false)
 |-- precipprob: integer (nullable = true)
 |-- preciptype: string (nullable = false)
 |-- snow: double (nullable = false)
 |-- snowdepth: double (nullable = false)
 |-- windgust: double (nullable = false)
 |-- windspeed: double (nullable = true)
 |-- winddir: double (nullable = true)
 |-- sealevelpressure: double (nullable = true)
 |-- cloudcover: double (nullable = true)
 |-- visibility: double (nullable = true)
 |-- severerisk: string (nullable = false)
 |-- conditions: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)



+-------------------+----+--------+------+----------+----------+----+---------+--------+---------+-------+----------------+----------+----------+----------+----------------+----------+----+
|           datetime|temp|humidity|precip|precipprob|preciptype|snow|snowdepth|windgust|windspeed|winddir|sealevelpressure|cloudcover|visibility|severerisk|      conditions|      date|hour|
+-------------------+----+--------+------+----------+----------+----+---------+--------+---------+-------+----------------+----------+----------+----------+----------------+----------+----+
|2025-12-08 08:00:00|29.9|   41.72|   0.0|         0|      None| 0.0|      0.0|    22.1|      9.2|  336.0|          1023.9|      23.9|       9.9|      None|Partially cloudy|2025-12-08|   8|
|2025-12-12 05:00:00|26.5|   42.76|   0.0|         0|      None| 0.0|      0.0|    21.5|      9.9|  292.0|          1014.0|       3.6|       9.9|      None|           Clear|2025-12-12|   5|
|2025-12-17 08:00:00|31.8|   57.99|   0.0|        

In [9]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("processing_3_World_Airports") \
    .getOrCreate()

# لود أول داتا ست
airports_df = spark.read.csv("/home/ahmed-refat/Desktop/flights & airports/Raw Data(batch)/World_Airports.csv", header=True, inferSchema=True)

airports_clean = airports_df \
    .select(
        "iata_code", "name", "type", "municipality",
        "iso_country", "iso_region", "latitude_deg",
        "longitude_deg", "elevation_ft", "scheduled_service"
    ) \
    .filter(F.col("iata_code").isNotNull()) \
    .dropDuplicates()

airports_clean.printSchema()
airports_clean.show(3)

root
 |-- iata_code: string (nullable = true)
 |-- name: string (nullable = true)
 |-- type: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- iso_country: string (nullable = true)
 |-- iso_region: string (nullable = true)
 |-- latitude_deg: double (nullable = true)
 |-- longitude_deg: double (nullable = true)
 |-- elevation_ft: integer (nullable = true)
 |-- scheduled_service: string (nullable = true)



+---------+--------------------+-------------+-------------+-----------+----------+-------------+--------------+------------+-----------------+
|iata_code|                name|         type| municipality|iso_country|iso_region| latitude_deg| longitude_deg|elevation_ft|scheduled_service|
+---------+--------------------+-------------+-------------+-----------+----------+-------------+--------------+------------+-----------------+
|      PQS|Pilot Station Air...|small_airport|Pilot Station|         US|     US-AK|    61.934601|   -162.899994|         305|              yes|
|      LVD|Lime Village Airport|small_airport| Lime Village|         US|     US-AK|61.3591003418|-155.440002441|         552|               no|
|      CTV|      Catoca Airport|small_airport|      Saurimo|         AO|    AO-LSU|    -9.430985|     20.311478|        3498|               no|
+---------+--------------------+-------------+-------------+-----------+----------+-------------+--------------+------------+-----------

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ==============================
# 1. إنشاء Spark Session
# ==============================
spark = SparkSession.builder \
    .appName("Gold Layer - Facts & Dimensions") \
    .getOrCreate()

# ==============================
# 2. تحميل البيانات من Silver (S3)
# ==============================
flights_df = spark.read.parquet("s3://your-bucket/silver/flights/")
weather_df = spark.read.parquet("s3://your-bucket/silver/weather/")
airports_df = spark.read.parquet("s3://your-bucket/silver/airports/")

# ==============================
# 3. بناء DIM_DATE
# ==============================
# بنبني الـ dimension من تواريخ الرحلات
dim_date = flights_df \
    .select(F.col("FL_DATE").alias("full_date")) \
    .distinct() \
    .withColumn("date_key", F.monotonically_increasing_id().cast("integer")) \
    .withColumn("day", F.dayofmonth("full_date")) \
    .withColumn("day_name", F.date_format("full_date", "EEEE")) \
    .withColumn("week_of_year", F.weekofyear("full_date")) \
    .withColumn("month", F.month("full_date")) \
    .withColumn("month_name", F.date_format("full_date", "MMMM")) \
    .withColumn("quarter", F.quarter("full_date")) \
    .withColumn("year", F.year("full_date")) \
    .withColumn("is_weekend", F.when(F.dayofweek("full_date").isin([1, 7]), 1).otherwise(0)) \
    .withColumn("season", F.when(F.month("full_date").isin([12, 1, 2]), "Winter")
                           .when(F.month("full_date").isin([3, 4, 5]), "Spring")
                           .when(F.month("full_date").isin([6, 7, 8]), "Summer")
                           .otherwise("Fall")) \
    .withColumn("holiday_flag", F.lit(0))  # تقدر تعدله لاحقاً وتحط فيه التواريخ الفعلية

# ==============================
# 4. بناء DIM_TIME
# ==============================
# بنبني الـ dimension من ساعات الرحلات
dim_time = flights_df \
    .select(F.col("CRS_DEP_TIME").alias("time_label")) \
    .distinct() \
    .withColumn("time_key", F.monotonically_increasing_id().cast("integer")) \
    .withColumn("hour", F.col("time_label").substr(1, 2).cast("integer")) \
    .withColumn("minute", F.col("time_label").substr(4, 2).cast("integer")) \
    .withColumn("part_of_day", F.when(F.col("hour").between(6, 11), "Morning")
                                 .when(F.col("hour").between(12, 17), "Afternoon")
                                 .when(F.col("hour").between(18, 21), "Evening")
                                 .otherwise("Night")) \
    .withColumn("time_slot_1h", F.concat(F.col("hour").cast("string"), F.lit(":00"))) \
    .withColumn("time_slot_3h", F.concat((F.col("hour") - (F.col("hour") % 3)).cast("string"), F.lit(":00"))) \
    .withColumn("peak_hour_flag", F.when(F.col("hour").between(7, 9) | F.col("hour").between(17, 19), 1).otherwise(0))

# ==============================
# 5. بناء DIM_AIRPORT
# ==============================
# بنبني الـ dimension من airports dataset
dim_airport = airports_df \
    .withColumn("airport_key", F.monotonically_increasing_id().cast("integer")) \
    .withColumnRenamed("name", "airport_name") \
    .withColumnRenamed("municipality", "city") \
    .withColumnRenamed("iso_country", "country") \
    .withColumnRenamed("iso_region", "state") \
    .withColumnRenamed("type", "airport_type") \
    .select(
        "airport_key", "iata_code", "airport_name", "city",
        "state", "country", "latitude_deg", "longitude_deg",
        "elevation_ft", "airport_type"
    )

# ==============================
# 6. بناء DIM_CARRIER
# ==============================
# بنبني الـ dimension من carriers الموجودين في flights
dim_carrier = flights_df \
    .select(F.col("OP_UNIQUE_CARRIER").alias("carrier_code")) \
    .distinct() \
    .withColumn("carrier_key", F.monotonically_increasing_id().cast("integer")) \
    .withColumn("carrier_name", F.lit(None).cast("string"))  # تقدر تضيف أسماء الشركات يدوياً لاحقاً

# ==============================
# 7. بناء DIM_WEATHER_CONDITION
# ==============================
# بنبني الـ dimension من conditions الموجودة في weather
dim_weather_condition = weather_df \
    .select("conditions", "preciptype", "severerisk") \
    .distinct() \
    .withColumn("weather_condition_key", F.monotonically_increasing_id().cast("integer")) \
    .withColumnRenamed("conditions", "weather_main") \
    .withColumn("weather_description", F.col("weather_main")) \
    .withColumnRenamed("preciptype", "precipitation_type") \
    .withColumnRenamed("severerisk", "severity_level")

# ==============================
# 8. بناء FACT_FLIGHT_DELAY
# ==============================

# أولاً بنعمل join بين flights وال dimensions عشان نجيب الـ keys
fact_flights = flights_df \
    .join(dim_date.select("date_key", "full_date"),
          flights_df["FL_DATE"] == dim_date["full_date"], "left") \
    .join(dim_time.select("time_key", "time_label").withColumnRenamed("time_key", "dep_time_key"),
          flights_df["CRS_DEP_TIME"] == dim_time["time_label"], "left") \
    .join(dim_airport.select("airport_key", "iata_code").withColumnRenamed("airport_key", "origin_airport_key"),
          flights_df["ORIGIN"] == dim_airport["iata_code"], "left") \
    .join(dim_airport.select("airport_key", "iata_code").withColumnRenamed("airport_key", "dest_airport_key"),
          flights_df["DEST"] == dim_airport["iata_code"], "left") \
    .join(dim_carrier.select("carrier_key", "carrier_code"),
          flights_df["OP_UNIQUE_CARRIER"] == dim_carrier["carrier_code"], "left")

# بعدين بنبني الـ fact table بالـ columns المطلوبة
fact_flight_delay = fact_flights \
    .withColumn("flight_delay_key", F.monotonically_increasing_id().cast("integer")) \
    .withColumn("flight_number", F.col("OP_CARRIER_FL_NUM").cast("string")) \
    .withColumn("delay_bucket",
                F.when(F.col("ARR_DELAY") <= 0, "No Delay")
                 .when(F.col("ARR_DELAY").between(1, 15), "Minor (1-15 min)")
                 .when(F.col("ARR_DELAY").between(16, 45), "Moderate (16-45 min)")
                 .when(F.col("ARR_DELAY").between(46, 120), "Severe (46-120 min)")
                 .otherwise("Critical (>120 min)")) \
    .withColumn("is_weather_related_flag",
                F.when(F.col("WEATHER_DELAY") > 0, 1).otherwise(0)) \
    .select(
        "flight_delay_key",
        "date_key",
        "dep_time_key",
        "origin_airport_key",
        "dest_airport_key",
        "carrier_key",
        "flight_number",
        F.col("FL_DATE").alias("fl_date"),
        F.col("DEP_DELAY").alias("dep_delay_minutes"),
        F.col("ARR_DELAY").alias("arr_delay_minutes"),
        F.col("CARRIER_DELAY").alias("carrier_delay_minutes"),
        F.col("WEATHER_DELAY").alias("weather_delay_minutes"),
        F.col("NAS_DELAY").alias("nas_delay_minutes"),
        F.col("SECURITY_DELAY").alias("security_delay_minutes"),
        F.col("LATE_AIRCRAFT_DELAY").alias("late_aircraft_delay_minutes"),
        F.col("CRS_ELAPSED_TIME").alias("scheduled_elapsed_time"),
        F.col("ACTUAL_ELAPSED_TIME").alias("actual_elapsed_time"),
        F.col("AIR_TIME").alias("air_time"),
        F.col("DISTANCE").alias("distance"),
        F.col("CANCELLED").alias("cancelled_flag"),
        F.col("DIVERTED").alias("diverted_flag"),
        F.col("IS_DEP_DELAYED").alias("is_dep_delayed_flag"),
        F.col("IS_ARR_DELAYED").alias("is_arr_delayed_flag"),
        "is_weather_related_flag",
        "delay_bucket"
    )

# ==============================
# 9. بناء FACT_WEATHER_OBSERVATION
# ==============================

# بنعمل join بين weather وال dimensions
fact_weather = weather_df \
    .join(dim_date.select("date_key", "full_date"),
          weather_df["date"] == dim_date["full_date"], "left") \
    .join(dim_time.select("time_key", "time_label"),
          weather_df["hour"] == F.col("time_label").substr(1, 2).cast("integer"), "left") \
    .join(dim_weather_condition.select("weather_condition_key", "weather_main"),
          weather_df["conditions"] == dim_weather_condition["weather_main"], "left")

fact_weather_observation = fact_weather \
    .withColumn("weather_observation_key", F.monotonically_increasing_id().cast("integer")) \
    .withColumn("airport_key", F.lit(None).cast("integer")) \ # مفيش airport_code في weather dataset هنربطه لاحقاً يدوياً
    .withColumn("weather_severity_score",
                F.when(F.col("severerisk") == "None", 0)
                 .when(F.col("severerisk") == "Low", 1)
                 .when(F.col("severerisk") == "Moderate", 2)
                 .when(F.col("severerisk") == "High", 3)
                 .otherwise(0)) \
    .select(
        "weather_observation_key",
        "date_key",
        "time_key",
        "airport_key",
        "weather_condition_key",
        F.col("temp").alias("temperature"),
        "humidity",
        F.col("precip").alias("precipitation"),
        "snow",
        F.col("windspeed").alias("wind_speed"),
        F.col("winddir").alias("wind_direction"),
        "visibility",
        F.col("sealevelpressure").alias("pressure"),
        "cloudcover",
        "weather_severity_score"
    )

# ==============================
# 10. حفظ الـ Gold Layer على S3
# ==============================

# حفظ الـ Dimensions
dim_date.write.mode("overwrite").parquet("s3://your-bucket/gold/dim_date/")
dim_time.write.mode("overwrite").parquet("s3://your-bucket/gold/dim_time/")
dim_airport.write.mode("overwrite").parquet("s3://your-bucket/gold/dim_airport/")
dim_carrier.write.mode("overwrite").parquet("s3://your-bucket/gold/dim_carrier/")
dim_weather_condition.write.mode("overwrite").parquet("s3://your-bucket/gold/dim_weather_condition/")

# حفظ الـ Facts
fact_flight_delay.write.mode("overwrite").parquet("s3://your-bucket/gold/fact_flight_delay/")
fact_weather_observation.write.mode("overwrite").parquet("s3://your-bucket/gold/fact_weather_observation/")

print("Gold Layer تم حفظه بنجاح على S3")